# Exercises — Financial trading with bt

[DataCamp exercise](https://campus.datacamp.com/courses/financial-trading-in-python/trading-basics-1?ex=9) · see `Notes.md` in this folder for the summary.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project's data folder regardless of where this notebook runs from
DATA = next(p / "course materials" / "data"
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "course materials" / "data").is_dir())

def load(name):
    """Load an OHLCV CSV with a parsed DatetimeIndex."""
    return pd.read_csv(DATA / name, index_col="Date", parse_dates=True)

def price(name, col, year=None):
    """Single-asset price DataFrame (column = `col`) for use with bt."""
    df = load(name)
    if year:
        df = df[df.index.year == year]
    return df["Close"].rename(col).to_frame()


### The 4-step bt process: get data, define strategy, backtest, evaluate

In [ ]:
import bt

# 1. Get data (equal-weight portfolio of three stocks, 2020 H1)
data = pd.DataFrame({
    "GOOG": load("GOOG-stock-data.csv")["Close"],
    "AMZN": load("AMZN-stock-data.csv")["Close"],
    "TSLA": load("TSLA-stock-data.csv")["Close"],
}).dropna()
data = data[(data.index.year == 2020) & (data.index <= "2020-06-30")]

# 2. Define the strategy from algos
bt_strategy = bt.Strategy("Trade_Weekly", [
    bt.algos.RunWeekly(),
    bt.algos.SelectAll(),
    bt.algos.WeighEqually(),
    bt.algos.Rebalance(),
])

# 3. Backtest
bt_result = bt.run(bt.Backtest(bt_strategy, data))

# 4. Evaluate
bt_result.plot(title="Backtest result")
plt.show()
bt_result.display()